In [2]:
import numpy as np
import pandas as pd

In [115]:
df = (
    pd
    .read_csv("01_flats.csv")
    .drop_duplicates()    #removing duplicate rows
    .dropna(subset=['price'])
    .apply(lambda col: col.str.strip().str.lower() if col.dtype=='object' else col)
    .assign(
        bhk = lambda df_: df_['property_name'].str.extract(r"(\d{1})").astype('int8')
        ,locality = lambda df_: df_['property_name'].str.split('in ').str[1]
        ,rating = lambda df_: df_['society'].str.extract(r"(\d{1}\.{1}\d+)").astype('float')
        ,society = lambda df_: df_['society'].str.split(r"(\d{1}\.{1}\d*)").str[0]
        ,price_cr = lambda df_: df_['price'].str.replace('₹','').apply(lambda x: x.replace('crore','').strip() if 'crore' in x else float(x.replace('lac','').strip())/100 if 'lac' in x else np.nan)
        ,bedRoom = lambda df_: df_['bedRoom'].str.extract(r"(\d+)")
        ,bathroom = lambda df_: df_['bathroom'].str.extract(r"(\d+)")
        ,balcony = lambda df_: df_['balcony'].str.extract(r"(\d*)")
        ,area_sqft = lambda df_: df_['areaWithType'].str.extract(r"(\d+)")
        ,area_sqm = lambda df_: df_['areaWithType'].str.extract(r"(\d+\.{1}\d+)")
        ,area_type = lambda df_: df_['areaWithType'].str.split(' area').str[0].str.strip()
        ,addl_room = lambda df_: np.where(df_['additionalRoom'].notna(), 1, 0)
        ,floor_no = lambda df_: df_['floorNum'].str.extract(r"(\d+)").fillna('0').astype(int)
        ,floor = lambda df_: np.where(df_['floor_no']<4, 'low rise', np.where(df_['floor_no']<8, 'mid rise', 'high rise'))
        ,age_1 = lambda df_: df_['agePossession'].str.replace('within 3 months','0').str.replace('within 6 months','0').str.findall(r"(\d+)").str[0].fillna(-1).astype(int).apply(lambda x: 2026-x if x>1000 else x)
        ,age = lambda df_: np.where(df_['age_1']<0, '-na-', np.where(df_['age_1']<2, 'new', np.where(df_['age_1']<5, 'old', 'ancient')))
        ,
    )
    .drop(columns=['property_name','link','property_id','price','area','areaWithType','additionalRoom','floorNum','floor_no','address','agePossession','age_1','description'])    #removing unnecessary columns
    .dropna(subset=['price_cr'])
)

print(df.shape)
df.sample(4)

(2996, 18)


,society,bedRoom,bathroom,balcony,facing,nearbyLocations,furnishDetails,features,rating,bhk,locality,price_cr,area_sqft,area_sqm,area_type,addl_room,floor,age
739,satya the hermitage,3,4,3,north-east,"['metro hospital, palam vihar', 'huda metro st...","['3 wardrobe', '5 fan', '1 exhaust fan', '2 ge...","['centrally air conditioned', 'water purifier'...",3.5,3,sector 103 gurgaon,0.995,1991,184.97,super built up,1,low rise,new
859,mahindra aura,3,3,3,north,"['global foyer mall, palam vihar', 'dwarka ex...","['3 wardrobe', '5 fan', '1 exhaust fan', '3 ge...","['water purifier', 'security / fire alarm', 'f...",4.2,3,sector 110 a gurgaon,1.4,1615,150.04,super built up,0,high rise,new
341,sare crescent parc royal greens phase 1,4,4,3,east,"['sri radhe krishna temple', 'standard charter...",NaN,"['security / fire alarm', 'lift(s)', 'maintena...",NaN,4,sector 92 gurgaon,1.25,2143,199.09,super built up,1,high rise,new
746,tulip white,3,2,3,west,"['sri radhe krishna temple', 'icici bank atm',...","['3 wardrobe', '4 fan', '1 exhaust fan', '2 ge...","['security / fire alarm', 'power back-up', 'fe...",3.5,3,sector 69 gurgaon,0.9,1500,139.35,super built up,0,low rise,new


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2996 entries, 0 to 3016
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   society          2995 non-null   object 
 1   bedRoom          2996 non-null   object 
 2   bathroom         2996 non-null   object 
 3   balcony          2996 non-null   object 
 4   additionalRoom   1692 non-null   object 
 5   address          2990 non-null   object 
 6   floorNum         2994 non-null   object 
 7   facing           2122 non-null   object 
 8   agePossession    2995 non-null   object 
 9   nearbyLocations  2905 non-null   object 
 10  description      2996 non-null   object 
 11  furnishDetails   2200 non-null   object 
 12  features         2589 non-null   object 
 13  rating           1701 non-null   float64
 14  bhk              2996 non-null   int8   
 15  locality         2996 non-null   object 
 16  price_cr         2996 non-null   object 
 17  area_sqft        29